# 02 — Preprocessing

Scale features to [0, 1] using statistics from the **normal-only** training
split, create validation splits, and prepare sliding windows for the LSTM
autoencoder. Cached outputs are saved to `data/processed/`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_hai, load_morris, morris_train_test_split
from src.preprocessing import prepare_tabular, make_windows, window_labels
from src.utils import set_seed, PROCESSED_DIR, save_figure

set_seed(42)
sns.set_theme(style='whitegrid', context='notebook')


## HAI

In [ ]:
hai_train, hai_test = load_hai()
# HAI is huge; subsample for speed in classical models but keep full for deep AE.
hai = prepare_tabular(hai_train, hai_test)
print('HAI X_train:', hai.X_train.shape)
print('HAI X_val  :', hai.X_val.shape)
print('HAI X_test :', hai.X_test.shape)
print('HAI y_test attack rate:', hai.y_test.mean().round(4))


In [ ]:
# Visualise scaling effect on a single sensor.
col_idx = 0
raw = hai_train.iloc[:, col_idx].to_numpy()[:5000]
scaled = hai.X_train[:5000, col_idx]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(raw, color='#2b8cbe', linewidth=0.6)
axes[0].set_title(f'Raw {hai.feature_names[col_idx]}')
axes[1].plot(scaled, color='#31a354', linewidth=0.6)
axes[1].set_title(f'MinMax-scaled {hai.feature_names[col_idx]}')
axes[1].set_ylim(-0.05, 1.05)
plt.tight_layout()
save_figure(fig, 'hai_scaling_effect', subdir='02_preprocessing')
plt.show()


In [ ]:
# Sliding windows for LSTM AE.
WINDOW = 60  # 60 s at 1 Hz
STRIDE = 10

hai_train_wins = make_windows(hai.X_train, WINDOW, STRIDE)
hai_val_wins   = make_windows(hai.X_val,   WINDOW, STRIDE)
hai_test_wins  = make_windows(hai.X_test,  WINDOW, STRIDE)
hai_test_wlabels = window_labels(hai.y_test, WINDOW, STRIDE)
print('HAI windows — train:', hai_train_wins.shape,
      '| val:', hai_val_wins.shape,
      '| test:', hai_test_wins.shape,
      '| attack rate:', hai_test_wlabels.mean().round(4))


In [ ]:
np.savez_compressed(
    PROCESSED_DIR / 'hai_tabular.npz',
    X_train=hai.X_train, X_val=hai.X_val,
    X_test=hai.X_test, y_test=hai.y_test,
)
np.savez_compressed(
    PROCESSED_DIR / 'hai_windows.npz',
    X_train=hai_train_wins, X_val=hai_val_wins,
    X_test=hai_test_wins, y_test=hai_test_wlabels,
)
print('Saved HAI processed arrays.')


## Morris

In [ ]:
morris = load_morris()
m_train, m_test = morris_train_test_split(morris)
morris_scaled = prepare_tabular(m_train, m_test)
print('Morris X_train:', morris_scaled.X_train.shape)
print('Morris X_val  :', morris_scaled.X_val.shape)
print('Morris X_test :', morris_scaled.X_test.shape)
print('Morris y_test attack rate:', morris_scaled.y_test.mean().round(4))


In [ ]:
# Morris packets arrive at irregular times, but ordered rows still carry
# sequence information — build windows to exercise the LSTM pipeline too.
MWINDOW = 30
MSTRIDE = 5
m_train_wins = make_windows(morris_scaled.X_train, MWINDOW, MSTRIDE)
m_val_wins   = make_windows(morris_scaled.X_val,   MWINDOW, MSTRIDE)
m_test_wins  = make_windows(morris_scaled.X_test,  MWINDOW, MSTRIDE)
m_test_wlabels = window_labels(morris_scaled.y_test, MWINDOW, MSTRIDE)
print('Morris windows — train:', m_train_wins.shape,
      '| val:', m_val_wins.shape,
      '| test:', m_test_wins.shape,
      '| attack rate:', m_test_wlabels.mean().round(4))

np.savez_compressed(
    PROCESSED_DIR / 'morris_tabular.npz',
    X_train=morris_scaled.X_train, X_val=morris_scaled.X_val,
    X_test=morris_scaled.X_test,   y_test=morris_scaled.y_test,
)
np.savez_compressed(
    PROCESSED_DIR / 'morris_windows.npz',
    X_train=m_train_wins, X_val=m_val_wins,
    X_test=m_test_wins,   y_test=m_test_wlabels,
)
print('Saved Morris processed arrays.')
